In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q datasets sentence-transformers

import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, pipeline


In [3]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
ds = Dataset.from_pandas(df)

ds = ds.map(lambda x: {'combined_text': x['prompt'] + ' ' + x['A']})

answer_q1 = len(ds[51]['combined_text'])
print(f"Q1 Answer: {answer_q1}")
print(f"Text preview: {ds[51]['combined_text'][:100]}...")


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1 Answer: 614
Text preview: Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what ...


In [4]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

answer_q2 = tokenizer.vocab_size
print(f"Q2 Answer: {answer_q2}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2 Answer: 30522


In [5]:
answer_q3 = tokenizer.convert_tokens_to_ids('[SEP]')
print(f"Q3 Answer: {answer_q3}")


Q3 Answer: 102


In [6]:
prompts = list(ds['prompt'])  # convert to list[str] — HF Dataset column is not a plain list

encoded = tokenizer(
    prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

answer_q4 = tuple(encoded['input_ids'].shape)
print(f"Q4 Answer: {answer_q4}")


Q4 Answer: (2000, 128)


In [7]:
answer_q5 = 768 // 12
print(f"Q5 Answer: {answer_q5}")


Q5 Answer: 64


In [8]:
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

row0_prompt = ds[0]['prompt']
print(f"Row 0 prompt: {row0_prompt}")

inputs = tokenizer(row0_prompt, return_tensors='pt')
print(f"Input IDs shape: {inputs['input_ids'].shape}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")

with torch.no_grad():
    outputs = model(**inputs)

answer_q6 = tuple(outputs.last_hidden_state.shape)
print(f"\nQ6 Answer: {answer_q6}")


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Row 0 prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Input IDs shape: torch.Size([1, 31])
Tokens: ['[CLS]', 'pick', 'the', 'best', 'possible', 'answer', ':', 'what', 'is', 'martin', 'he', '##ide', '##gger', "'", 's', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', '?', 'among', 'the', 'listed', 'options', '.', '[SEP]']

Q6 Answer: (1, 31, 768)


In [9]:
cls_vector = outputs.last_hidden_state[0, 0, :]  # [CLS] is token index 0

first_5 = cls_vector[:5]
print(f"First 5 values: {first_5.tolist()}")

answer_q7 = round(first_5.sum().item(), 4)
print(f"\nQ7 Answer: {answer_q7}")


First 5 values: [-0.46766412258148193, -0.07544441521167755, -0.20190182328224182, -0.007064236328005791, -0.4480222463607788]

Q7 Answer: -1.2001


In [10]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs_attn = tokenizer(text, return_tensors='pt')

tokens = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
print(f"Tokens: {tokens}")

# Find the index of 'fusion'
fusion_idx = tokens.index('fusion')
print(f"'fusion' is at token index: {fusion_idx}")

with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

# Last layer attention, first head
# attentions shape: tuple of (batch, num_heads, seq_len, seq_len)
last_layer_attn = outputs_attn.attentions[-1]       # last layer
first_head_attn = last_layer_attn[0, 0, :, :]       # batch 0, head 0

# [CLS] (index 0) attending to 'fusion'
answer_q8 = round(first_head_attn[0, fusion_idx].item(), 4)
print(f"\nQ8 Answer: {answer_q8}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
'fusion' is at token index: 4

Q8 Answer: 0.1025


In [11]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

row0_prompt = ds[0]['prompt']
row0_B      = ds[0]['B']

print(f"Prompt: {row0_prompt[:100]}...")
print(f"Option B: {row0_B[:100]}...")

emb_prompt = st_model.encode(row0_prompt)
emb_B      = st_model.encode(row0_B)

sim = util.cos_sim(emb_prompt, emb_B)
answer_q9 = round(sim.item(), 4)
print(f"\nQ9 Answer: {answer_q9}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and ...
Option B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relation...

Q9 Answer: 0.7658


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cos_sim

options = ['A', 'B', 'C', 'D', 'E']

def map_at_3(truth, preds):
    score, hits = 0.0, 0
    for i, p in enumerate(preds[:3]):
        if p == truth:
            hits += 1
            score += hits / (i + 1)
    return score

# ── Pipeline 1: TF-IDF ──
all_texts = []
for i in range(len(ds)):
    row = ds[i]
    for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
        all_texts.append(str(row[col]))

tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(all_texts)

tfidf_top3 = []
for i in range(len(ds)):
    row = ds[i]
    p_vec = tfidf.transform([row['prompt']])
    sims = {}
    for o in options:
        o_vec = tfidf.transform([str(row[o])])
        sims[o] = sklearn_cos_sim(p_vec, o_vec)[0][0]
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    tfidf_top3.append(ranked)

print("TF-IDF pipeline done.")

# ── Pipeline 2: MiniLM (BATCHED for speed) ──
# Encode ALL texts at once instead of one-by-one
print("Encoding all prompts + options in batch...")

all_prompts = [ds[i]['prompt'] for i in range(len(ds))]
all_options  = {o: [str(ds[i][o]) for i in range(len(ds))] for o in options}

# Batch encode — this is ~50x faster than individual calls
emb_prompts = st_model.encode(all_prompts, show_progress_bar=True, batch_size=64)
emb_options = {}
for o in options:
    emb_options[o] = st_model.encode(all_options[o], show_progress_bar=False, batch_size=64)
    print(f"  Encoded option {o}")

# Compute similarities from cached embeddings
minilm_top3 = []
minilm_scores = []

for i in range(len(ds)):
    sims = {}
    for o in options:
        sims[o] = util.cos_sim(emb_prompts[i], emb_options[o][i]).item()
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    minilm_top3.append(ranked)
    minilm_scores.append(map_at_3(ds[i]['answer'], ranked))

# MAP@3 for MiniLM
minilm_map3 = np.mean(minilm_scores)
print(f"\nMiniLM MAP@3: {round(minilm_map3, 4)}")

# Count: NOT in TF-IDF top-3 BUT IS in MiniLM top-3
count = 0
for i in range(len(ds)):
    truth = ds[i]['answer']
    if truth not in tfidf_top3[i] and truth in minilm_top3[i]:
        count += 1

print(f"Count (not in TF-IDF but in MiniLM): {count}")
print(f"\nQ10 Answer: MAP@3 = {round(minilm_map3, 4)}, Count = {count}")


TF-IDF pipeline done.
Encoding all prompts + options in batch...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  Encoded option A
  Encoded option B
  Encoded option C
  Encoded option D
  Encoded option E

MiniLM MAP@3: 0.4231
Count (not in TF-IDF but in MiniLM): 462

Q10 Answer: MAP@3 = 0.4231, Count = 462


In [13]:
classifier = pipeline("zero-shot-classification")

row1 = ds[1]
prompt_1 = row1['prompt']
labels = [row1['A'], row1['B'], row1['C']]

print(f"Prompt: {prompt_1[:100]}...")
print(f"Labels (A,B,C): {[l[:50]+'...' for l in labels]}")

result_softmax = classifier(prompt_1, candidate_labels=labels)

print(f"\nRankings:")
for label, score in zip(result_softmax['labels'], result_softmax['scores']):
    print(f"  {score:.4f}: {label[:60]}...")

answer_q11 = round(result_softmax['scores'][0], 4)
print(f"\nQ11 Answer: {answer_q11}")
print(f"Sum of 3 probs (softmax): {sum(result_softmax['scores']):.4f}")


No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Prompt: What is accelerator-based light-ion fusion?...
Labels (A,B,C): ['Accelerator-based light-ion fusion is a technique ...', 'Accelerator-based light-ion fusion is a technique ...', 'Accelerator-based light-ion fusion is a technique ...']

Rankings:
  0.4575: Accelerator-based light-ion fusion is a technique that uses ...
  0.2751: Accelerator-based light-ion fusion is a technique that uses ...
  0.2675: Accelerator-based light-ion fusion is a technique that uses ...

Q11 Answer: 0.4575
Sum of 3 probs (softmax): 1.0000


In [14]:
result_sigmoid = classifier(prompt_1, candidate_labels=labels, multi_label=True)

print(f"Multi-label (Sigmoid) rankings:")
for label, score in zip(result_sigmoid['labels'], result_sigmoid['scores']):
    print(f"  {score:.4f}: {label[:60]}...")

sum_softmax = sum(result_softmax['scores'])
sum_sigmoid = sum(result_sigmoid['scores'])

answer_q12 = round(abs(sum_softmax - sum_sigmoid), 4)
print(f"\nSum softmax: {sum_softmax:.4f}")
print(f"Sum sigmoid: {sum_sigmoid:.4f}")
print(f"\nQ12 Answer: {answer_q12}")


Multi-label (Sigmoid) rankings:
  0.0005: Accelerator-based light-ion fusion is a technique that uses ...
  0.0000: Accelerator-based light-ion fusion is a technique that uses ...
  0.0000: Accelerator-based light-ion fusion is a technique that uses ...

Sum softmax: 1.0000
Sum sigmoid: 0.0005

Q12 Answer: 0.9995


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Get first sample
row0 = ds[0]

prompt = f"""
Question: {row0['prompt']}

A. {row0['A']}
B. {row0['B']}

Answer with only A or B.
"""

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt")

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False
)

# Decode
answer_q13 = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print("Q13 Prediction:", answer_q13)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13 Prediction: B


In [16]:
print("=" * 60)
print("  MILESTONE 2 — ANSWER SUMMARY")
print("=" * 60)
print(f"  Q1  combined_text length [51]:     {answer_q1}")
print(f"  Q2  BERT vocab size:               {answer_q2}")
print(f"  Q3  [SEP] token ID:                {answer_q3}")
print(f"  Q4  input_ids shape:               {answer_q4}")
print(f"  Q5  Attention head dim:            {answer_q5}")
print(f"  Q6  last_hidden_state shape:       {answer_q6}")
print(f"  Q7  Sum of first 5 CLS values:     {answer_q7}")
print(f"  Q8  Attention [CLS]→fusion:        {answer_q8}")
print(f"  Q9  Cosine sim (prompt, B):        {answer_q9}")
print(f"  Q10 MiniLM MAP@3 / Count:          {round(minilm_map3, 4)} / {count}")
print(f"  Q11 Top zero-shot prob (softmax):  {answer_q11}")
print(f"  Q12 |sum_softmax - sum_sigmoid|:   {answer_q12}")

print("=" * 60)


  MILESTONE 2 — ANSWER SUMMARY
  Q1  combined_text length [51]:     614
  Q2  BERT vocab size:               30522
  Q3  [SEP] token ID:                102
  Q4  input_ids shape:               (2000, 128)
  Q5  Attention head dim:            64
  Q6  last_hidden_state shape:       (1, 31, 768)
  Q7  Sum of first 5 CLS values:     -1.2001
  Q8  Attention [CLS]→fusion:        0.1025
  Q9  Cosine sim (prompt, B):        0.7658
  Q10 MiniLM MAP@3 / Count:          0.4231 / 462
  Q11 Top zero-shot prob (softmax):  0.4575
  Q12 |sum_softmax - sum_sigmoid|:   0.9995
